In [ ]:
from jormungandr.fafnir import Fafnir
from jormungandr.backbone import Backbone
from jormungandr.dataset import create_dataloaders
import torch
import supervision as sv
from supervision.metrics import MeanAveragePrecision


In [ ]:
fafnir = Fafnir()

In [ ]:
train_loader, val_loader = create_dataloaders()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
map_metric = MeanAveragePrecision()

with torch.no_grad():
    for batch in val_loader:
        pixel_values, pixel_mask, labels = (
                batch["pixel_values"],
                batch["pixel_mask"],
                batch["labels"],
            )
        pixel_values = pixel_values.to(device)
        pixel_mask = pixel_mask.to(device)
        labels = [{k: v.to(device) for k, v in label.items()} for label in labels]

        class_labels, bbox_coordinates = fafnir.forward(pixel_values)
        
        print(f"Class lables shape: {class_labels.shape}")
        print(bbox_coordinates.shape)
        print(class_labels.argmax())
        pred_class = class_labels.argmax(dim=-1)   # shape: [32, 100]
        
        # Use HungerinanMatcher to find 
        # Example: keep only real detections for one image b
        print(pred_class)
        map_metric.update(pred_class, labels)
        
        break
                

In [ ]:
map_result = map_metric.compute()

print(map_result.map50_95)
# 0.4674

print(map_result)
# MeanAveragePrecisionResult:
# Metric target: MetricTarget.BOXES
# Class agnostic: False
# mAP @ 50:95: 0.4674
# mAP @ 50:    0.5048
# mAP @ 75:    0.4796
# mAP scores: [0.50485  0.50377  0.50377  ...]
# IoU thresh: [0.5  0.55  0.6  ...]
# AP per class:
# 0: [0.67699  0.67699  0.67699  ...]
# ...
# Small objects: ...
# Medium objects: ...
# Large objects: ...

map_result.plot()